[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lugerlakes/signal_analysis_course/blob/main/Lab3_Caracterizacion_emisor.ipynb)

<div style="border-left:6px solid #1F4E79;padding:10px 16px;background:#F3F7FB">

<span style="font-size:12px;letter-spacing:1px;color:#1F4E79">CURSO DE GUERRA ELECTRÓNICA · ASIGNATURA: ANÁLISIS DE SEÑALES DE RADAR</span>

# Laboratorio 3 · Caracterización de un emisor
### PRI, exploración de antena, agilidad en frecuencia y cambios de modo

<span style="font-size:12px;color:#7a7a7a">MATERIAL DE INSTRUCCIÓN · DATOS SINTÉTICOS · Emisores y bibliotecas ficticios</span>
</div>

| | |
|---|---|
| ⏱️ **Duración estimada** | 4 horas (2 de guía + 2 de trabajo por parejas) |
| 📥 **Datos de entrada** | `D2_ESM_capturas_caracterizacion.xlsx` (8 capturas en hojas `CAP_01` … `CAP_08`) |
| 📤 **Producto al final** | Ficha de caracterización de la captura asignada a tu pareja |

**Objetivos de aprendizaje** — al terminar este laboratorio serás capaz de:

1. Reconocer los **tipos de modulación de PRI** (estable, jitter, stagger, deslizante, dwell & switch) en sus dos gráficos firma.
2. Reconocer los **tipos de exploración** (circular, sectorial, cónica, fija/lock-on) a partir de la amplitud vs tiempo.
3. Medir **periodo de scan, tiempo en blanco y ancho de haz** de la antena del radar.
4. Detectar **agilidad en frecuencia** y **cambios de modo** (búsqueda → seguimiento).
5. Calcular parámetros derivados (PRF, R<sub>ua</sub>, ΔR, ciclo de trabajo, impactos por barrido) y formular una **hipótesis de función**.
6. Elaborar la **ficha de caracterización** de un emisor.

**Cómo leer este cuaderno**

| Ícono | Significado |
|---|---|
| 📘 | Concepto o teoría breve |
| ▶️ | Celda de código para **ejecutar** (clic en la celda y `Shift + Enter`) |
| ✏️ | **Tu turno**: completa el código o responde |
| 💡 | Consejo práctico |
| ⚠️ | Error frecuente o precaución |
| 🎯 | Conexión con la Guerra Electrónica (ESM / ELINT) |

> Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla de forma extraña: menú **Entorno de ejecución → Reiniciar y ejecutar todo**.

---
## 0. Preparación
La caja de herramientas incluye ahora lo del Lab 1 (carga/limpieza), lo del Lab 2 (estadística) y las nuevas funciones de **caracterización** que construiremos paso a paso en este laboratorio.

In [ ]:
# @title 🧰 Caja de herramientas (Labs 1–3) — ejecutar
# ------------------------------------------------------------------
# A. CARGA Y LIMPIEZA DE PDW (lo que aprendiste a mano en el Lab 1)
# ------------------------------------------------------------------
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 4)

CARPETAS_DATOS = [".", "datos", "/content", "/content/datos", "/content/sample_data",
                  "/content/drive/MyDrive", "/content/drive/MyDrive/datos",
                  "/content/drive/MyDrive/Analisis_de_Señales"]


def buscar_archivo(nombre):
    """Busca 'nombre' en las carpetas habituales (local y Google Colab)."""
    for carpeta in CARPETAS_DATOS:
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    raise FileNotFoundError(
        f"No encontré '{nombre}'. Súbelo con el ícono de carpeta 📁 (panel izquierdo de Colab) "
        f"o copia su ruta exacta en la variable del archivo.")


def estandarizar_columnas(df):
    """Renombra las columnas de la exportación ESM a nombres estándar con unidades."""
    nuevos = {}
    for col in df.columns:
        c = str(col).upper()
        if c.startswith("UNNAMED") or c.strip() == "":
            continue
        if "TOA" in c:
            nuevos[col] = "TOA_us"
        elif "FREQ" in c or "FREC" in c:
            nuevos[col] = "FREQ_MHz"
        elif c.startswith("PW") or "WIDTH" in c:
            nuevos[col] = "PW_us"
        elif "AMP" in c:
            nuevos[col] = "AMP_dBm"
        elif "AOA" in c or "DOA" in c or "BEARING" in c:
            nuevos[col] = "AOA_deg"
        elif "PRI" in c:
            nuevos[col] = "PRI_us"
    columnas_vacias = [c for c in df.columns if c not in nuevos]
    return df.drop(columns=columnas_vacias).rename(columns=nuevos)


def limpiar_pdw(df, verbose=True):
    """Limpieza estándar de PDW: centinelas, vacíos, duplicados y orden por TOA."""
    n0 = len(df)
    df = df.copy()
    claves = [c for c in ["TOA_us", "FREQ_MHz", "PW_us"] if c in df.columns]
    df = df.dropna(subset=claves)
    n1 = len(df)
    malos = (df["FREQ_MHz"] <= 0) | (df["PW_us"] <= 0)      # registros "fantasma" del receptor
    df = df[~malos]
    n2 = len(df)
    df = df.drop_duplicates()
    n3 = len(df)
    df = df.sort_values("TOA_us").reset_index(drop=True)
    df["TOA_s"] = (df["TOA_us"] - df["TOA_us"].iloc[0]) / 1e6
    df["DTOA_us"] = df["TOA_us"].diff()
    if verbose:
        print(f"Filas originales ............ {n0:>7,}")
        print(f"  − con celdas vacías ....... {n0 - n1:>7,}")
        print(f"  − con ceros / centinela ... {n1 - n2:>7,}")
        print(f"  − duplicadas .............. {n2 - n3:>7,}")
        print(f"Filas válidas ............... {n3:>7,}")
    return df


# Carpeta web con los datos del curso (repositorio público). Déjala vacía ("") para trabajar sólo con archivos locales.
URL_DATOS = "https://raw.githubusercontent.com/lugerlakes/signal_analysis_course/main/datos/"


def asegurar_archivo(nombre):
    """Devuelve la ruta del archivo; si no está, lo descarga (URL_DATOS) o pide subirlo (Colab)."""
    try:
        return buscar_archivo(nombre)
    except FileNotFoundError:
        pass
    if URL_DATOS:
        import urllib.request, urllib.parse
        os.makedirs("datos", exist_ok=True)
        destino = os.path.join("datos", nombre)
        try:
            urllib.request.urlretrieve(URL_DATOS.rstrip("/") + "/" + urllib.parse.quote(nombre), destino)
            print(f"⬇️ '{nombre}' descargado desde el repositorio del curso")
            return destino
        except Exception as e:
            print(f"⚠️ No se pudo descargar '{nombre}' ({e}). Intentando otras opciones…")
            if os.path.exists(destino):
                os.remove(destino)
    try:
        from google.colab import files
        print(f"📁 No encuentro '{nombre}'. Selecciónalo desde tu computador:")
        subidos = files.upload()
        if nombre in subidos:
            return nombre
    except ImportError:
        pass
    return buscar_archivo(nombre)   # mensaje de error claro


def cargar_pdw(nombre_archivo, hoja=0, verbose=True):
    """Carga un archivo ESM (Excel o CSV), estandariza columnas y limpia."""
    ruta = asegurar_archivo(nombre_archivo)
    if ruta.lower().endswith(".csv"):
        crudo = pd.read_csv(ruta)
    else:
        crudo = pd.read_excel(ruta, sheet_name=hoja)
    df = limpiar_pdw(estandarizar_columnas(crudo), verbose=verbose)
    if verbose:
        dur = df["TOA_s"].iloc[-1]
        print(f"Duración de la captura ...... {dur:,.2f} s")
    return df
# ------------------------------------------------------------------
# B. ESTADÍSTICA DESCRIPTIVA (Lab 2)
# ------------------------------------------------------------------
def moda_continua(serie, bins="auto"):
    """Moda de una variable continua = centro de la barra más alta del histograma."""
    x = pd.Series(serie).dropna().values
    cuentas, bordes = np.histogram(x, bins=bins)
    i = np.argmax(cuentas)
    return 0.5 * (bordes[i] + bordes[i + 1])


def ic95_media(serie, confianza=0.95):
    """Intervalo de confianza para la MEDIA usando la distribución t de Student."""
    x = pd.Series(serie).dropna().values
    n = len(x)
    media = x.mean()
    ee = x.std(ddof=1) / np.sqrt(n)              # error estándar de la media
    t = stats.t.ppf(0.5 + confianza / 2, df=n - 1)
    return media - t * ee, media + t * ee


def ic95_mediana_bootstrap(serie, n_boot=2000, confianza=0.95, semilla=0):
    """Intervalo de confianza para la MEDIANA por remuestreo (bootstrap)."""
    x = pd.Series(serie).dropna().values
    rng = np.random.default_rng(semilla)
    medianas = np.median(rng.choice(x, size=(n_boot, len(x)), replace=True), axis=1)
    a = (1 - confianza) / 2
    return np.quantile(medianas, a), np.quantile(medianas, 1 - a)


def resumen_estadistico(serie, nombre=None):
    """Tabla con los estadígrafos más usados en un informe de análisis."""
    x = pd.Series(serie).dropna()
    lo, hi = ic95_media(x)
    return pd.Series({
        "n": int(len(x)),
        "media": x.mean(),
        "IC95_media_inf": lo,
        "IC95_media_sup": hi,
        "mediana": x.median(),
        "moda (histograma)": moda_continua(x),
        "desv_estandar": x.std(ddof=1),
        "CV_%": 100 * x.std(ddof=1) / abs(x.mean()) if x.mean() != 0 else np.nan,
        "minimo": x.min(),
        "P5": x.quantile(0.05),
        "Q1 (P25)": x.quantile(0.25),
        "Q3 (P75)": x.quantile(0.75),
        "P95": x.quantile(0.95),
        "maximo": x.max(),
        "rango": x.max() - x.min(),
        "IQR": x.quantile(0.75) - x.quantile(0.25),
        "asimetria": stats.skew(x),
    }, name=nombre or getattr(serie, "name", "variable"))


def graficar_distribucion(serie, titulo="", unidad="", bins=60):
    """Histograma + KDE, boxplot y violín de una misma variable (3 vistas)."""
    x = pd.Series(serie).dropna()
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw={"width_ratios": [2.2, 1, 1]})
    sns.histplot(x, bins=bins, kde=True, ax=ax[0], color="#4C72B0")
    ax[0].axvline(x.mean(), color="#C44E52", ls="--", lw=1.5, label=f"media = {x.mean():.4g}")
    ax[0].axvline(x.median(), color="#55A868", ls="-", lw=1.5, label=f"mediana = {x.median():.4g}")
    ax[0].legend(fontsize=9)
    ax[0].set_title(f"Histograma + KDE · {titulo}")
    ax[0].set_xlabel(unidad)
    sns.boxplot(y=x, ax=ax[1], color="#8172B2", width=0.4)
    ax[1].set_title("Boxplot"); ax[1].set_ylabel(unidad)
    sns.violinplot(y=x, ax=ax[2], color="#64B5CD", inner="quartile", cut=0)
    ax[2].set_title("Violín"); ax[2].set_ylabel(unidad)
    plt.tight_layout()
    plt.show()
# ------------------------------------------------------------------
# C. CARACTERIZACIÓN DE UN EMISOR (Lab 3)
# ------------------------------------------------------------------
from scipy.signal import find_peaks
from collections import Counter

C_LUZ = 299_792_458.0  # m/s


def banda_ieee(f_mhz):
    f = f_mhz / 1000.0
    for lim, b in [(0.3, "VHF"), (1, "UHF"), (2, "L"), (4, "S"), (8, "C"), (12, "X"),
                   (18, "Ku"), (27, "K"), (40, "Ka"), (75, "V"), (110, "W")]:
        if f < lim:
            return b
    return "mm"


def banda_otan(f_mhz):
    f = f_mhz / 1000.0
    for lim, b in [(0.25, "A"), (0.5, "B"), (1, "C"), (2, "D"), (3, "E"), (4, "F"), (6, "G"),
                   (8, "H"), (10, "I"), (20, "J"), (40, "K"), (60, "L"), (100, "M")]:
        if f < lim:
            return b
    return ">M"


def analizar_pri(toa_us, verbose=True):
    """Analiza la secuencia de llegadas de UN emisor y sugiere el tipo de PRI.

    Pasos: (1) DTOA = diferencias de TOA; (2) PRI base = valor más bajo frecuente;
    (3) se descartan huecos de iluminación y pulsos perdidos (DTOA > 1.8·PRI base);
    (4) se buscan niveles discretos y se mide la correlación pulso a pulso."""
    toa = np.sort(np.asarray(toa_us, float))
    d = np.diff(toa)
    d = d[d > 0]
    r = {"n_dtoa": len(d)}
    if len(d) < 10:
        r["tipo"] = "Insuficientes pulsos"
        return r
    # (2) PRI base sobre escala logarítmica (bins de 0.5 %)
    ld = np.log(d)
    lb = np.arange(ld.min(), np.quantile(ld, 0.98) + 0.01, 0.005)
    if len(lb) < 3:
        lb = np.linspace(ld.min(), ld.max() + 1e-6, 5)
    cnt, _ = np.histogram(ld, bins=lb)
    acum = np.convolve(cnt, np.ones(7), mode="same")
    i0 = int(np.argmax(acum >= 0.03 * len(d)))
    base = float(np.exp(lb[i0]))
    principal = (d >= 0.8 * base) & (d <= 1.8 * base)
    m = d[principal]
    r.update(pri_base=base, fraccion_principal=principal.mean(),
             fraccion_perdidos=(d > 1.8 * base).mean())
    med = np.median(m)
    r.update(pri_mediana=med, pri_media=m.mean(), pri_min=np.quantile(m, 0.01),
             pri_max=np.quantile(m, 0.99), pri_std=m.std(ddof=1),
             pri_cv_pct=100 * m.std(ddof=1) / m.mean())
    # (4a) niveles discretos
    ancho = max(0.003 * med, 0.05)
    bordes = np.arange(m.min() - ancho, m.max() + 2 * ancho, ancho)
    h, _ = np.histogram(m, bins=bordes)
    centros = 0.5 * (bordes[1:] + bordes[:-1])
    picos, _ = find_peaks(np.r_[0, h, 0], height=0.08 * h.max(), distance=3)
    picos = picos - 1
    niveles = []
    for p in picos:
        sel = np.abs(m - centros[p]) <= 1.5 * ancho
        if sel.sum():
            niveles.append(float(np.median(m[sel])))
    niveles = sorted(niveles)
    conc = np.mean(np.min(np.abs(m[:, None] - np.array(niveles)[None, :]), axis=1) <= 2 * ancho) if niveles else 0
    discreto = (1 <= len(niveles) <= 8) and conc >= 0.85
    # (4b) correlación entre PRI consecutivos (sólo pares sin pulsos perdidos)
    ok = principal[:-1] & principal[1:]
    a, b = d[:-1][ok], d[1:][ok]
    r1 = float(np.corrcoef(a, b)[0, 1]) if len(a) > 5 and a.std() > 0 and b.std() > 0 else 0.0
    r.update(niveles=niveles, concentracion=conc, autocorr_lag1=r1)
    # (5) clasificación
    if r["fraccion_principal"] < 0.4:
        r["tipo"] = "Sin estructura (posible ruido o mezcla)"
        r["detalle"] = (f"sólo {100 * r['fraccion_principal']:.0f} % de los DTOA forman un patrón: "
                        "no hay un tren de pulsos coherente")
        if verbose:
            print(f"Tipo de PRI sugerido : {r['tipo']}\nDetalle              : {r['detalle']}")
        return r
    if discreto and len(niveles) == 1:
        tipo = "Estable"
        detalle = f"PRI ≈ {niveles[0]:.2f} µs (PRF ≈ {1e6 / niveles[0]:.1f} Hz), CV = {r['pri_cv_pct']:.3f} %"
    elif discreto:
        niv = np.array(niveles)
        idx = np.argmin(np.abs(m[:, None] - niv[None, :]), axis=1)
        corridas = np.diff(np.flatnonzero(np.r_[True, idx[1:] != idx[:-1], True]))
        largo_corrida = corridas.mean()
        if largo_corrida >= 4:
            tipo = "Dwell & Switch"
            detalle = (f"{len(niv)} PRI: " + ", ".join(f"{v:.1f}" for v in niv) +
                       f" µs · ~{largo_corrida:.0f} pulsos por permanencia (dwell)")
            r["pulsos_por_dwell"] = largo_corrida
        else:
            tipo = f"Stagger {len(niv)} posiciones"
            k = len(niv)
            ventanas = [tuple(idx[i:i + k]) for i in range(len(idx) - k)]
            ventanas = [w for w in ventanas if len(set(w)) == k]
            seq = Counter(ventanas).most_common(1)[0][0] if ventanas else tuple(range(k))
            orden = [niv[j] for j in seq]
            r["secuencia"] = orden
            r["periodo_trama"] = float(niv.sum())
            detalle = ("secuencia " + " → ".join(f"{v:.1f}" for v in orden) +
                       f" µs · periodo de trama {niv.sum():.1f} µs · PRI media {niv.mean():.1f} µs")
    else:
        if r1 > 0.6:
            tipo = "Deslizante (sliding)"
            caidas = np.flatnonzero(np.diff(m) < -0.5 * (r["pri_max"] - r["pri_min"]))
            pulsos_rampa = float(np.median(np.diff(caidas))) if len(caidas) > 2 else np.nan
            r["pulsos_por_rampa"] = pulsos_rampa
            detalle = (f"rampa de {r['pri_min']:.0f} a {r['pri_max']:.0f} µs, ~{pulsos_rampa:.0f} pulsos por rampa")
        else:
            tipo = "Jitter (aleatorio)"
            pct = 100 * (r["pri_max"] - r["pri_min"]) / (2 * med)
            r["jitter_pct"] = pct
            detalle = f"PRI central ≈ {med:.1f} µs, variación ≈ ±{pct:.1f} %"
    r["tipo"] = tipo
    r["detalle"] = detalle
    if verbose:
        print(f"Tipo de PRI sugerido : {tipo}")
        print(f"Detalle              : {detalle}")
        print(f"Pulsos perdidos      : {100 * r['fraccion_perdidos']:.1f} % de los DTOA (múltiplos de la PRI o huecos de scan)")
    return r


def graficar_pri(toa_us, titulo="", max_pulsos=600):
    """Dos vistas clásicas: secuencia DTOA vs nº de pulso, e histograma de DTOA."""
    d = np.diff(np.sort(np.asarray(toa_us, float)))
    lim = np.quantile(d, 0.97) * 1.3
    fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
    n = min(len(d), max_pulsos)
    ax[0].plot(np.arange(n), d[:n], ".", ms=4)
    ax[0].set_ylim(0, lim)
    ax[0].set_xlabel("Nº de pulso"); ax[0].set_ylabel("DTOA (µs)")
    ax[0].set_title(f"Secuencia de PRI · {titulo}")
    ax[1].hist(d[d < lim], bins=200, color="#4C72B0")
    ax[1].set_xlabel("DTOA (µs)"); ax[1].set_ylabel("Nº de pulsos")
    ax[1].set_title("Histograma de DTOA")
    plt.tight_layout(); plt.show()


def detectar_iluminaciones(df, gap_us=100_000, min_pulsos=5):
    """Agrupa pulsos en 'iluminaciones' (ráfagas) separadas por huecos > gap_us.
    Se ignoran grupos con menos de 'min_pulsos' (pulsos sueltos de lóbulos laterales)."""
    toa = df["TOA_us"].values
    tsec = df["TOA_s"].values if "TOA_s" in df else toa / 1e6     # segundos desde el inicio de la captura
    amp = df["AMP_dBm"].values
    corte = np.r_[0, np.flatnonzero(np.diff(toa) > gap_us) + 1, len(toa)]
    filas = []
    for i, j in zip(corte[:-1], corte[1:]):
        if j - i < min_pulsos:
            continue
        t, a = toa[i:j], amp[i:j]
        sel = a >= a.max() - 3.0                       # pulsos dentro de 3 dB del máximo
        w = 10 ** (a[sel] / 10)
        # ancho del lóbulo principal: ajuste de una parábola (en dB) a los pulsos
        # que están a menos de 10 dB del máximo  →  ancho a -3 dB = 2·sqrt(3/|c2|)
        lob = a >= a.max() - 10.0
        tot = np.nan
        if lob.sum() >= 5:
            tc = (t[lob] - t[lob].mean()) / 1e3                      # ms
            c2 = np.polyfit(tc, a[lob], 2)[0]
            if c2 < 0:
                tot = 2 * np.sqrt(3.0 / -c2)
        ts = tsec[i:j]
        filas.append({
            "t_inicio_s": ts[0], "t_fin_s": ts[-1], "n_pulsos": j - i,
            "duracion_ms": (t[-1] - t[0]) / 1e3,
            "t_centro_s": np.sum(ts[sel] * w) / np.sum(w),
            "amp_max_dBm": a.max(),
            "tot_3dB_ms": tot,
        })
    return pd.DataFrame(filas)


def espectro_amplitud(df, fmax=200.0):
    """Espectro (FFT) de la amplitud: revela modulaciones periódicas (scan cónico)."""
    t = df["TOA_us"].values / 1e6
    a = df["AMP_dBm"].values
    fs = min(1.0 / np.median(np.diff(t)), 4 * fmax)
    tu = np.arange(t[0], t[-1], 1 / fs)
    au = np.interp(tu, t, a)
    au = (au - au.mean()) * np.hanning(len(au))
    esp = np.abs(np.fft.rfft(au)) / len(au)
    f = np.fft.rfftfreq(len(au), 1 / fs)
    sel = (f >= 2) & (f <= fmax)
    return f[sel], esp[sel]


def analizar_scan(df, gap_us=100_000, verbose=True):
    """Clasifica el tipo de exploración de antena a partir de AMP vs TOA."""
    il = detectar_iluminaciones(df, gap_us)
    r = {"n_iluminaciones": len(il)}
    durac_total = (df["TOA_us"].iloc[-1] - df["TOA_us"].iloc[0]) / 1e6
    if len(il) == 0:
        r.update(tipo="Indeterminado", periodo_s=np.nan, detalle="pulsos dispersos, sin iluminaciones definidas")
        if verbose:
            print(f"Tipo de scan sugerido : {r['tipo']}\nDetalle               : {r['detalle']}")
        return r
    if len(il) <= 2 and il["duracion_ms"].max() / 1e3 > 0.6 * durac_total:
        f, e = espectro_amplitud(df)
        k = int(np.argmax(e))
        prominencia = e[k] / (np.median(e) + 1e-12)
        prof = df["AMP_dBm"].quantile(0.95) - df["AMP_dBm"].quantile(0.05)
        if prominencia > 8 and prof > 1.5:
            r.update(tipo="Cónico", frecuencia_scan_Hz=float(f[k]), periodo_s=1 / float(f[k]),
                     profundidad_dB=float(prof),
                     detalle=f"modulación de amplitud a {f[k]:.1f} Hz (periodo {1000 / f[k]:.1f} ms), profundidad ≈ {prof:.1f} dB")
        else:
            r.update(tipo="Fijo (iluminación continua)", periodo_s=0.0,
                     detalle=f"amplitud estable (variación {prof:.1f} dB): posible SEGUIMIENTO / lock-on")
    elif len(il) == 2:
        dt2 = float(il["t_centro_s"].iloc[1] - il["t_centro_s"].iloc[0])
        r.update(tipo="Indeterminado (2 iluminaciones)", periodo_s=dt2,
                 detalle=f"sólo 2 iluminaciones separadas {dt2:.2f} s: periodo probable, BAJA confianza (captura más larga)")
    elif len(il) < 3:
        r.update(tipo="Indeterminado", periodo_s=np.nan, detalle="una sola iluminación: no se puede estimar el periodo")
    else:
        dt = np.diff(il["t_centro_s"].values)
        cv = dt.std() / dt.mean()
        if cv < 0.08:
            T = float(np.median(dt))
            tot = float(il["tot_3dB_ms"].median())
            haz = 360.0 * tot / 1000 / T
            r.update(tipo="Circular", periodo_s=T, rpm=60 / T, tot_3dB_ms=tot, haz_estimado_deg=haz,
                     detalle=f"periodo {T:.2f} s ({60 / T:.1f} rpm) · tiempo en blanco (3 dB) {tot:.1f} ms · haz ≈ {haz:.1f}°")
        else:
            pares = dt[:-1] + dt[1:]
            alterna = np.mean(np.sign(np.diff(dt)[:-1]) != np.sign(np.diff(dt)[1:])) > 0.7
            if alterna and pares.std() / pares.mean() < 0.08:
                T = float(np.median(pares))
                r.update(tipo="Sectorial (bidireccional)", periodo_s=T,
                         detalle=f"iluminaciones en pares alternos ({np.median(dt[::2]):.2f} s / {np.median(dt[1::2]):.2f} s) · periodo de barrido {T:.2f} s")
            else:
                r.update(tipo="Irregular", periodo_s=float(np.median(dt)),
                         detalle="intervalos irregulares: posible cambio de modo, exploración electrónica o varios emisores mezclados")
        largas = il["duracion_ms"] > 5 * il["duracion_ms"].median()
        if largas.any():
            r["detalle"] += f" · ⚠ {int(largas.sum())} iluminación(es) prolongada(s): posible paso a SEGUIMIENTO"
    if verbose:
        print(f"Tipo de scan sugerido : {r['tipo']}")
        print(f"Detalle               : {r['detalle']}")
    return r


def media_circular(angulos_deg):
    """Media de ángulos respetando el salto 359° → 0°."""
    a = np.deg2rad(np.asarray(angulos_deg, float))
    return float(np.rad2deg(np.arctan2(np.sin(a).mean(), np.cos(a).mean())) % 360)


def ficha_emisor(df, nombre="EMISOR", gap_us=100_000, verbose=True):
    """Ficha paramétrica completa de un emisor (RF, PW, PRI, scan, AOA)."""
    f = df["FREQ_MHz"]
    rango_rf = f.quantile(0.99) - f.quantile(0.01)
    agil = rango_rf > 15
    pri = analizar_pri(df["TOA_us"], verbose=False)
    scan = analizar_scan(df, gap_us=gap_us, verbose=False)
    ficha = {
        "EMISOR": nombre,
        "N_PULSOS": len(df),
        "T_INICIO_s": df["TOA_s"].min() if "TOA_s" in df else np.nan,
        "T_FIN_s": df["TOA_s"].max() if "TOA_s" in df else np.nan,
        "RF_MEDIANA_MHz": round(f.median(), 1),
        "RF_MIN_MHz": round(f.quantile(0.01), 1),
        "RF_MAX_MHz": round(f.quantile(0.99), 1),
        "RF_TIPO": "Ágil" if agil else "Fija",
        "BANDA_IEEE": banda_ieee(f.median()),
        "BANDA_OTAN": banda_otan(f.median()),
        "PW_MEDIANA_us": round(df["PW_us"].median(), 2),
        "PW_P5_us": round(df["PW_us"].quantile(0.05), 2),
        "PW_P95_us": round(df["PW_us"].quantile(0.95), 2),
        "PRI_TIPO": pri.get("tipo"),
        "PRI_DETALLE": pri.get("detalle"),
        "PRI_MIN_us": round(pri.get("pri_min", np.nan), 1),
        "PRI_MAX_us": round(pri.get("pri_max", np.nan), 1),
        "PRI_MEDIA_us": round(pri.get("pri_media", np.nan), 1),
        "PRF_MEDIA_Hz": round(1e6 / pri.get("pri_media", np.nan), 1),
        "SCAN_TIPO": scan.get("tipo"),
        "SCAN_PERIODO_s": round(scan.get("periodo_s", np.nan), 3),
        "SCAN_DETALLE": scan.get("detalle"),
        "AOA_MEDIA_deg": round(media_circular(df["AOA_deg"]), 1) if "AOA_deg" in df else np.nan,
        "AMP_MAX_dBm": round(df["AMP_dBm"].max(), 1),
    }
    if verbose:
        for k, v in ficha.items():
            print(f"{k:<16}: {v}")
    return ficha
print('Caja de herramientas lista ✔')

---
## 1. Teoría breve

### 1.1 Modulaciones de PRI y para qué sirven
| Tipo | Qué hace el radar | Propósito típico | Firma en **DTOA vs nº de pulso** | Firma en **histograma** |
|---|---|---|---|---|
| **Estable** | PRI constante | Simplicidad; radares de búsqueda clásicos | Línea horizontal | Un pico muy angosto |
| **Jitter** | PRI varía al azar (±1 a ±15 %) | Protección electrónica (EP): dificulta engaño en distancia y sincronización del jammer | Nube horizontal sin patrón | Meseta rectangular |
| **Stagger** (*n* posiciones) | Secuencia fija de *n* PRI que se repite | Eliminar **velocidades ciegas** del MTI | Escalones que se repiten cada *n* pulsos | *n* picos de igual altura |
| **Deslizante** (*sliding*) | PRI crece/decrece en rampa y se reinicia | Cubrir distintas altitudes/alcances (radares 3D) | Dientes de sierra | Meseta continua |
| **Dwell & Switch** | Mantiene una PRI durante un grupo de pulsos y cambia | Resolver **ambigüedades** en radares pulso-Doppler | Escalones **largos** | Varios picos |

🎯 La modulación de PRI es una de las **huellas digitales** más útiles para identificar un emisor, y su **función**.

▶️ Veamos cómo lucen en versión ideal (sin ruido ni pulsos perdidos):

In [ ]:
n = np.arange(180)
ideales = {
    "Estable": np.full(180, 1000.0),
    "Jitter ±10 %": 1000 * (1 + np.random.default_rng(0).uniform(-0.1, 0.1, 180)),
    "Stagger 3 posiciones": np.array([1000, 1150, 1300])[n % 3].astype(float),
    "Deslizante": 700 + 300 * (n % 30) / 29,
    "Dwell & Switch": np.array([700, 850, 1000])[(n // 60) % 3].astype(float),
}
fig, ax = plt.subplots(2, 5, figsize=(18, 5.5))
for j, (nombre, pri) in enumerate(ideales.items()):
    ax[0, j].plot(n, pri, ".", ms=3); ax[0, j].set_ylim(600, 1400); ax[0, j].set_title(nombre)
    ax[1, j].hist(pri, bins=60, range=(600, 1400), color="#55A868")
    ax[0, j].set_xlabel("Nº de pulso"); ax[1, j].set_xlabel("PRI (µs)")
ax[0, 0].set_ylabel("PRI (µs)"); ax[1, 0].set_ylabel("Nº de pulsos")
plt.tight_layout(); plt.show()

### 1.2 Tipos de exploración (*scan*) y su firma en la amplitud
El receptor ESM "ve" el diagrama de radiación del radar pasar frente a él. La **amplitud vs tiempo** revela cómo se mueve la antena:

| Scan | Firma en AMP vs tiempo | Qué se mide | Función típica |
|---|---|---|---|
| **Circular** | Ráfagas iguales y **equiespaciadas** | Periodo (s), rpm, tiempo en blanco, ancho de haz | Vigilancia 360° |
| **Sectorial** (bidireccional) | Ráfagas en **pares** con intervalos alternos corto/largo | Periodo de barrido, ancho de sector | Adquisición, vigilancia de un sector |
| **Cónico** | Amplitud continua con **modulación senoidal** | Frecuencia de scan (Hz) por FFT | Seguimiento angular (tiro) |
| **Fijo** (*lock-on*) | Amplitud **continua y constante** | — | **Seguimiento de un blanco: ¡posible amenaza sobre la plataforma!** |
| Electrónico (antena en fase) | Iluminaciones irregulares, cortas | Tiempos de permanencia | Multifunción |

**Parámetros derivados del scan (circular):**
$$\text{rpm} = \frac{60}{T_{scan}} \qquad \theta_{3dB} \approx 360° \cdot \frac{T_{iluminación, 3dB}}{T_{scan}} \qquad \text{impactos por barrido} \approx T_{iluminación, 3dB}\cdot PRF$$

In [ ]:
t = np.linspace(0, 12, 6000)          # 12 s
tc = np.linspace(0, 0.2, 6000)        # 0,2 s (zoom para el cónico)
def rafagas(t, centros, ancho):
    return np.max([np.exp(-0.5 * ((t - c) / ancho) ** 2) for c in centros], axis=0)
firmas = [
    ("Circular (T = 4 s)", t, -30 + 30 * rafagas(t, [1, 5, 9], 0.04)),
    ("Sectorial (T = 4 s)", t, -30 + 30 * rafagas(t, [1, 2.2, 5, 6.2, 9, 10.2], 0.04)),
    ("Cónico 35 Hz (zoom de 0,2 s)", tc, -2 - 2 * np.cos(2 * np.pi * 35 * tc)),
    ("Fijo / lock-on", t, -1 + 0.2 * np.random.default_rng(1).normal(size=t.size)),
]
fig, ax = plt.subplots(1, 4, figsize=(18, 3.2))
for a, (nombre, x, y) in zip(ax, firmas):
    a.plot(x, y, lw=1); a.set_title(nombre); a.set_xlabel("tiempo (s)"); a.set_ylim(-33, 3)
ax[0].set_ylabel("Amplitud relativa (dB)")
plt.tight_layout(); plt.show()

---
## 2. Galería: 8 capturas, 8 emisores distintos

📘 El archivo tiene **8 hojas**. Con `sheet_name=None`, pandas devuelve un **diccionario** `{nombre_hoja: DataFrame}`, y lo recorremos con un `for` para limpiar cada captura.

In [ ]:
ruta = asegurar_archivo("D2_ESM_capturas_caracterizacion.xlsx")
hojas = pd.read_excel(ruta, sheet_name=None)          # diccionario de DataFrames
capturas = {nombre: limpiar_pdw(estandarizar_columnas(h), verbose=False) for nombre, h in hojas.items()}

for nombre, d in capturas.items():
    print(f"{nombre}: {len(d):6,} pulsos · {d['TOA_s'].iloc[-1]:6.2f} s · RF mediana {d['FREQ_MHz'].median():8.1f} MHz")

In [ ]:
fig, ax = plt.subplots(8, 2, figsize=(16, 24))
for i, (nombre, d) in enumerate(capturas.items()):
    dt = np.diff(d["TOA_us"].values)
    k = min(len(dt), 400)
    lim = np.quantile(dt[:k], 0.9) * 1.6
    ax[i, 0].plot(np.arange(k), dt[:k], ".", ms=3)
    ax[i, 0].set_ylim(0, lim); ax[i, 0].set_ylabel(f"{nombre}\nDTOA (µs)")
    ax[i, 1].plot(d["TOA_s"], d["AMP_dBm"], ".", ms=1.5, color="tab:red")
    ax[i, 1].set_ylabel("AMP (dBm)")
ax[0, 0].set_title("Secuencia de DTOA (primeros 400 pulsos)"); ax[0, 1].set_title("Amplitud vs tiempo (s)")
ax[-1, 0].set_xlabel("Nº de pulso"); ax[-1, 1].set_xlabel("Tiempo (s)")
plt.tight_layout(); plt.show()

### ✏️ Tu turno 1 · Clasificación visual
Usando las tablas de la sección 1, completa la tabla **sólo mirando los gráficos** (después lo confirmaremos con números).

| Captura | Tipo de PRI | Tipo de scan | Observaciones |
|---|---|---|---|
| CAP_01 | | | |
| CAP_02 | | | |
| CAP_03 | | | |
| CAP_04 | | | |
| CAP_05 | | | |
| CAP_06 | | | |
| CAP_07 | | | (pista: mira la RF) |
| CAP_08 | | | (pista: mira el final) |

💡 Los puntos sueltos por encima de la secuencia principal son **pulsos perdidos** (DTOA = suma de dos o más PRI).

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


| Captura | PRI | Scan | Observaciones |
|---|---|---|---|
| CAP_01 | Estable (~800 µs) | Circular (~2,5 s) | Banda X, PW corto: navegación/superficie |
| CAP_02 | Jitter (~450 µs ± 7 %) | Sectorial (pares de ráfagas) | Banda C: adquisición |
| CAP_03 | Stagger 3 posiciones | Circular lento (~10 s) | Banda S, PW 2 µs: alerta temprana |
| CAP_04 | Deslizante (sliding) | Circular (~5 s) | PW 4 µs: radar 3D |
| CAP_05 | Dwell & Switch (3 PRF altas) | Fijo (continuo) | Pulso-Doppler en seguimiento |
| CAP_06 | Estable, PRF alta (5 kHz) | Cónico (modulación senoidal) | Banda Ku: seguimiento de tiro |
| CAP_07 | Estable (~600 µs) | Circular (~4 s) | **Agilidad en frecuencia** |
| CAP_08 | Estable, **cambia** | Circular → **Fijo** | **Cambio de modo búsqueda → seguimiento** |


</details>

---
## 3. Ejemplo resuelto paso a paso: CAP_03

### 3.1 Vista general

In [ ]:
cap = capturas["CAP_03"]
fig, ax = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
ax[0].plot(cap["TOA_s"], cap["AMP_dBm"], ".", ms=2, color="tab:red"); ax[0].set_ylabel("AMP (dBm)")
ax[1].plot(cap["TOA_s"], cap["FREQ_MHz"], ".", ms=2); ax[1].set_ylabel("RF (MHz)")
ax[2].plot(cap["TOA_s"], cap["PW_us"], ".", ms=2, color="tab:orange"); ax[2].set_ylabel("PW (µs)")
ax[2].set_xlabel("Tiempo (s)"); ax[0].set_title(f"CAP_03 · {len(cap):,} pulsos")
plt.tight_layout(); plt.show()

### 3.2 RF y PW
🎯 Criterio práctico: si el rango P1–P99 de la RF supera unos **15 MHz** (varias veces la precisión del receptor), sospecha **agilidad en frecuencia**.

In [ ]:
fuertes = cap[cap["AMP_dBm"] > cap["AMP_dBm"].max() - 20]       # pulsos a menos de 20 dB del máximo
tabla = pd.concat([resumen_estadistico(fuertes["FREQ_MHz"], "RF (MHz)"),
                   resumen_estadistico(fuertes["PW_us"], "PW (µs)")], axis=1).T
display(tabla[["n", "mediana", "IC95_media_inf", "IC95_media_sup", "desv_estandar", "P5", "P95"]].round(3))

rf = cap["FREQ_MHz"]
rango = rf.quantile(0.99) - rf.quantile(0.01)
print(f"Rango P1–P99 de la RF: {rango:.1f} MHz → {'ÁGIL' if rango > 15 else 'FIJA'}")
print(f"Banda IEEE {banda_ieee(rf.median())} · banda OTAN {banda_otan(rf.median())}")

### 3.3 PRI paso a paso
**Paso A — DTOA y su histograma (escala log para ver todo a la vez):**

In [ ]:
d = np.diff(cap["TOA_us"].values)
plt.figure(figsize=(13, 3.5))
plt.hist(d, bins=np.logspace(np.log10(d.min()), np.log10(d.max()), 300))
plt.xscale("log"); plt.xlabel("DTOA (µs, escala log)"); plt.ylabel("Nº")
plt.title("Todos los DTOA: PRI, pulsos perdidos y huecos de scan")
plt.show()

Se distinguen tres "familias": las **PRI** (≈ 1000–1300 µs), sus **sumas** por pulsos perdidos (≈ 2000–4000 µs) y los **huecos entre iluminaciones** (≈ 10⁷ µs = 10 s).

**Paso B — quedarnos con la PRI.** Tomamos la PRI base (el menor valor frecuente) y descartamos todo DTOA mayor que 1,8 veces ese valor (pulsos perdidos y huecos):

In [ ]:
base = np.quantile(d, 0.02)            # aproximación simple de la PRI más corta
principal = d[(d >= 0.8 * base) & (d <= 1.8 * base)]
print(f"PRI base ≈ {base:.1f} µs · DTOA conservados: {len(principal)} de {len(d)} ({100*len(principal)/len(d):.0f} %)")

plt.figure(figsize=(13, 3.5))
plt.hist(principal, bins=300); plt.xlabel("DTOA (µs)"); plt.ylabel("Nº")
plt.title("DTOA 'principales' (sin pulsos perdidos ni huecos)"); plt.show()

**Paso C — niveles y secuencia.** Hay **3 peaks de igual altura**. ¿Stagger o Dwell & Switch? La diferencia está en la **secuencia**: en el stagger el nivel cambia **en cada pulso**; en el D&S se repite durante muchos pulsos seguidos.

In [ ]:
niveles = np.array([1000, 1150, 1300])                 # leídos del histograma
idx = np.argmin(np.abs(principal[:, None] - niveles[None, :]), axis=1)
print("18 niveles consecutivos (mitad de la captura):", niveles[idx[600:618]])
cambios = np.mean(idx[1:] != idx[:-1])
print(f"El nivel cambia en el {100*cambios:.0f} % de los pulsos → {'STAGGER' if cambios > 0.5 else 'DWELL & SWITCH'}")

**Paso D — la función automática.** La caja de herramientas hace todo lo anterior (con algunos refinamientos: PRI base robusta, detección automática de peaks, correlación entre pulsos consecutivos para separar *jitter* de *sliding*):

In [ ]:
graficar_pri(cap["TOA_us"], "CAP_03")
r_pri = analizar_pri(cap["TOA_us"])

In [ ]:
print("Parámetros derivados por cada posición del stagger:")
for p in r_pri["niveles"]:
    print(f"  PRI {p:7.1f} µs → PRF {1e6/p:7.1f} Hz · R_ua {C_LUZ*p*1e-6/2/1000:6.1f} km")
pw = fuertes["PW_us"].median()
print(f"Resolución en distancia (pulso sin modulación): {C_LUZ*pw*1e-6/2:.0f} m")
print(f"Ciclo de trabajo medio: {100*pw/r_pri['pri_media']:.3f} %")

### 3.4 Exploración de antena paso a paso
**Paso A — separar iluminaciones:** agrupamos los pulsos en "ráfagas" separadas por huecos de más de 100 ms.

In [ ]:
il = detectar_iluminaciones(cap, gap_us=100_000)
il.round(3)

In [ ]:
T = np.diff(il["t_centro_s"].values)
print("Tiempo entre iluminaciones (s):", np.round(T, 3))
print(f"Periodo de scan ≈ {np.median(T):.2f} s  →  {60/np.median(T):.1f} rpm")

**Paso B — tiempo en blanco y ancho de haz.** El lóbulo principal en dB tiene forma de **parábola invertida**. Ajustamos una parábola a los pulsos que están a menos de 10 dB del máximo; su ancho a −3 dB es el **tiempo de iluminación a 3 dB**. Con él y el periodo estimamos el **ancho del haz**.

In [ ]:
k = 1                                              # nº de iluminación a examinar
ini, fin = il.loc[k, "t_inicio_s"], il.loc[k, "t_fin_s"]
r = cap[(cap["TOA_s"] >= ini - 0.01) & (cap["TOA_s"] <= fin + 0.01)]
lob = r[r["AMP_dBm"] >= r["AMP_dBm"].max() - 10]
t0 = lob["TOA_s"].mean()
c2, c1, c0 = np.polyfit((lob["TOA_s"] - t0) * 1000, lob["AMP_dBm"], 2)     # tiempo en ms
tt = np.linspace(lob["TOA_s"].min(), lob["TOA_s"].max(), 200)
ajuste = c2 * ((tt - t0) * 1000) ** 2 + c1 * (tt - t0) * 1000 + c0
ancho_ms = 2 * np.sqrt(3 / -c2)
t_vertice = t0 - c1 / (2 * c2) / 1000

plt.figure(figsize=(12, 4))
plt.plot(r["TOA_s"], r["AMP_dBm"], ".", ms=4, label="pulsos")
plt.plot(tt, ajuste, "r-", lw=2, label="parábola ajustada (lóbulo principal)")
plt.axhline(ajuste.max() - 3, color="gray", ls="--", label="−3 dB")
plt.axvspan(t_vertice - ancho_ms / 2000, t_vertice + ancho_ms / 2000, alpha=0.2, color="orange",
            label=f"tiempo a −3 dB = {ancho_ms:.1f} ms")
plt.xlabel("Tiempo (s)"); plt.ylabel("AMP (dBm)"); plt.legend(); plt.title("Una iluminación del haz")
plt.show()

Tscan = np.median(T)
print(f"Ancho de haz estimado θ3dB ≈ 360° × {ancho_ms:.1f} ms / {Tscan*1000:.0f} ms = {360*ancho_ms/1000/Tscan:.2f}°")
print(f"Impactos por barrido (a −3 dB) ≈ {ancho_ms/1000 * 1e6/r_pri['pri_media']:.0f} pulsos")

**Paso C — función automática:**

In [ ]:
r_scan = analizar_scan(cap)

### 3.5 Ficha del emisor y evaluación

In [ ]:
ficha03 = ficha_emisor(cap, "CAP_03")

🎯 **Hipótesis de función (CAP_03):** banda S, PW 2 µs, PRI ≈ 1–1,3 ms (R<sub>ua</sub> ≈ 150–195 km), **stagger** (MTI sin velocidades ciegas), scan circular lento (10 s = 6 rpm) y haz angosto en azimut (~1,5°) ⇒ **radar de vigilancia aérea / alerta temprana de largo alcance**. Amenaza directa: **baja** (no es un radar de tiro), pero indica que la plataforma **está siendo detectada**.

---
## 4. Casos especiales

### 4.1 Scan cónico (CAP_06): la FFT de la amplitud
📘 La **Transformada Rápida de Fourier (FFT)** descompone una señal en sus frecuencias. Si la amplitud sube y baja periódicamente (la antena gira su haz alrededor del blanco), la FFT muestra un **pico en la frecuencia de scan**.

In [ ]:
c6 = capturas["CAP_06"]
f, esp = espectro_amplitud(c6)
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
z = c6[c6["TOA_s"] < 0.2]
ax[0].plot(z["TOA_s"] * 1000, z["AMP_dBm"], ".-", ms=2, lw=0.5)
ax[0].set_xlabel("Tiempo (ms)"); ax[0].set_ylabel("AMP (dBm)"); ax[0].set_title("Amplitud (primeros 200 ms)")
ax[1].plot(f, esp); ax[1].set_xlabel("Frecuencia (Hz)"); ax[1].set_ylabel("Magnitud")
ax[1].set_title(f"Espectro de la amplitud · pico en {f[np.argmax(esp)]:.1f} Hz")
plt.tight_layout(); plt.show()
analizar_scan(c6);

### 4.2 Agilidad en frecuencia (CAP_07)

In [ ]:
c7 = capturas["CAP_07"]
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
ax[0].plot(c7["TOA_s"], c7["FREQ_MHz"], ".", ms=2); ax[0].set_xlabel("Tiempo (s)"); ax[0].set_ylabel("RF (MHz)")
ax[0].set_title("RF vs tiempo")
ax[1].hist(c7["FREQ_MHz"], bins=200); ax[1].set_xlabel("RF (MHz)"); ax[1].set_title("Histograma de RF")
plt.tight_layout(); plt.show()

canales = np.sort(c7["FREQ_MHz"].round(-1).value_counts().loc[lambda s: s > 10].index.values)
saltos = c7["FREQ_MHz"].diff().abs()
print(f"Canales detectados: {len(canales)} entre {canales.min():.0f} y {canales.max():.0f} MHz · separación ≈ {np.median(np.diff(canales)):.0f} MHz")
print(f"Pulsos consecutivos que cambian de canal (> 5 MHz): {100*(saltos > 5).mean():.0f} % → agilidad PULSO A PULSO")

🎯 La **agilidad en frecuencia** es una técnica de **protección electrónica (EP)**: un jammer de ruido "puntual" (*spot*) deja de ser efectivo, obligando a interferir **toda la banda** (*barrage*) con menos potencia por MHz. En el informe se reporta: rango de RF, nº de canales, separación y tipo de salto (pulso a pulso / por ráfagas).

### 4.3 Cambio de modo (CAP_08): búsqueda → seguimiento
📘 Una **mediana móvil** (`rolling`) calcula la mediana en una ventana que se desliza por los datos: suaviza el ruido y deja ver **cambios de comportamiento**.

In [ ]:
c8 = capturas["CAP_08"].copy()
c8["PW_movil"] = c8["PW_us"].rolling(31, center=True).median()
c8["DTOA_movil"] = c8["DTOA_us"].rolling(31, center=True).median()

fig, ax = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
ax[0].plot(c8["TOA_s"], c8["AMP_dBm"], ".", ms=2, color="tab:red"); ax[0].set_ylabel("AMP (dBm)")
ax[1].plot(c8["TOA_s"], c8["PW_movil"], ".", ms=2); ax[1].set_ylabel("PW mediana móvil (µs)")
ax[2].plot(c8["TOA_s"], c8["DTOA_movil"], ".", ms=2, color="tab:green"); ax[2].set_ylabel("DTOA mediana móvil (µs)")
ax[2].set_xlabel("Tiempo (s)")
plt.tight_layout(); plt.show()

t_cambio = c8.loc[c8["PW_movil"] < 1.0, "TOA_s"].min()
print(f"Instante del cambio de modo ≈ {t_cambio:.2f} s desde el inicio de la captura")

In [ ]:
modo_busq = c8[c8["TOA_s"] < t_cambio]
modo_seg = c8[c8["TOA_s"] >= t_cambio]
fichas8 = pd.DataFrame([ficha_emisor(modo_busq, "CAP_08 · BÚSQUEDA", verbose=False),
                        ficha_emisor(modo_seg, "CAP_08 · SEGUIMIENTO", verbose=False)]).set_index("EMISOR")
fichas8[["N_PULSOS", "RF_MEDIANA_MHz", "PW_MEDIANA_us", "PRI_TIPO", "PRI_MEDIA_us", "PRF_MEDIA_Hz", "SCAN_TIPO", "SCAN_PERIODO_s"]].T

🎯 **Interpretación:** la misma RF y el mismo AOA (mismo emisor) pasan de **scan circular, PRI 1500 µs, PW 2 µs** a **iluminación continua, PRI 250 µs (PRF 4 kHz), PW 0,4 µs**. Es la secuencia típica **búsqueda → adquisición → seguimiento (lock-on)**: el radar dejó de buscar y está **siguiendo un blanco**. Si el receptor está en la plataforma propia, esto es una **alerta de amenaza** que debe reportarse de inmediato con la **hora exacta del cambio**.

---
## 5. Trabajo por parejas: tu captura

| Pareja | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| Captura | CAP_01 | CAP_02 | CAP_03 | CAP_04 | CAP_05 | CAP_06 | CAP_07 | CAP_08 |

▶️ Cambia el valor de `CAPTURA` y ejecuta las celdas en orden. *(La pareja de CAP_03 debe repetir el análisis sin mirar el ejemplo.)*

In [ ]:
CAPTURA = "CAP_01"        # ✏️ cambia por la captura de tu pareja
mi = capturas[CAPTURA]
print(f"{CAPTURA}: {len(mi):,} pulsos en {mi['TOA_s'].iloc[-1]:.2f} s")

In [ ]:
# Paso 1 · Vista general
fig, ax = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
ax[0].plot(mi["TOA_s"], mi["AMP_dBm"], ".", ms=2, color="tab:red"); ax[0].set_ylabel("AMP (dBm)")
ax[1].plot(mi["TOA_s"], mi["FREQ_MHz"], ".", ms=2); ax[1].set_ylabel("RF (MHz)")
ax[2].plot(mi["TOA_s"], mi["PW_us"], ".", ms=2, color="tab:orange"); ax[2].set_ylabel("PW (µs)")
ax[2].set_xlabel("Tiempo (s)"); ax[0].set_title(CAPTURA)
plt.tight_layout(); plt.show()

In [ ]:
# Paso 2 · RF y PW (estadística del Lab 2, con pulsos a menos de 20 dB del máximo)
f_mi = mi[mi["AMP_dBm"] > mi["AMP_dBm"].max() - 20]
graficar_distribucion(f_mi["FREQ_MHz"], "RF", "MHz")
graficar_distribucion(f_mi["PW_us"], "PW", "µs")
pd.concat([resumen_estadistico(f_mi["FREQ_MHz"], "RF (MHz)"), resumen_estadistico(f_mi["PW_us"], "PW (µs)")], axis=1).T.round(3)

In [ ]:
# Paso 3 · PRI
graficar_pri(mi["TOA_us"], CAPTURA)
r_mi = analizar_pri(mi["TOA_us"])

In [ ]:
# Paso 4 · Scan (si la iluminación es continua, mira también el espectro de amplitud)
s_mi = analizar_scan(mi)
f_esp, e_esp = espectro_amplitud(mi)
plt.figure(figsize=(10, 3)); plt.plot(f_esp, e_esp); plt.xlabel("Frecuencia (Hz)"); plt.title("Espectro de la amplitud"); plt.show()

In [ ]:
# Paso 5 · Ficha completa
ficha_mi = ficha_emisor(mi, CAPTURA)

### ✏️ Tu turno 2 · Ficha de caracterización (completa y presenta en 3 minutos)

| Campo | Valor |
|---|---|
| Captura / pareja | |
| Nº de pulsos y duración | |
| RF (mediana, IC95, P5–P95) · banda IEEE / OTAN · fija o ágil | |
| PW (mediana, P5–P95) | |
| Tipo de PRI y valores (µs) · PRF (Hz) | |
| R<sub>ua</sub> · ΔR · ciclo de trabajo | |
| Tipo de scan · periodo · rpm | |
| Tiempo en blanco (3 dB) · ancho de haz estimado · impactos por barrido | |
| Eventos (cambio de modo, agilidad, etc.) con hora | |
| **Hipótesis de función** (vigilancia, adquisición, seguimiento, tiro, navegación…) | |
| **Nivel de confianza** (alto / medio / bajo) y por qué | |

💡 **CAP_08:** antes de la ficha, detecta el cambio de modo (sección 4.3) y haz **una ficha por modo**. **CAP_05 y CAP_06:** el scan es continuo; usa el espectro de amplitud.

---
## 6. Preguntas de repaso
1. ¿Por qué un radar con MTI usa stagger? ¿Qué problema tendría con PRI estable?
2. Dos emisores tienen la misma RF y PRI, pero uno muestra iluminaciones cada 10 s y otro amplitud constante. ¿Cuál te preocupa más y por qué?
3. Si la ESM pierde muchos pulsos, ¿qué le pasa al histograma de DTOA? ¿Afecta la estimación de la PRI?
4. ¿Qué limitación tiene estimar el ancho de haz con un receptor que está en el lóbulo lateral (señal débil)?

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. Con PRI estable el MTI tiene **velocidades ciegas** (blancos cuya velocidad Doppler es múltiplo de la PRF se cancelan). Variando la PRI (stagger), esas velocidades cambian de un pulso a otro y se eliminan los "puntos ciegos".
2. El de **amplitud constante**: indica iluminación continua (**lock-on / seguimiento**), típico de una fase previa al empleo de armas. El de 10 s está en búsqueda.
3. Aparecen peaks en **múltiplos y sumas** de la PRI y baja la altura relativa de los peaks verdaderos. La PRI se sigue estimando bien si se toma el **nivel más bajo** y se descartan los DTOA > 1,8·PRI base (como hace `analizar_pri`).
4. El ajuste necesita ver el **lóbulo principal** con suficiente margen sobre el umbral; si sólo vemos lóbulos laterales o el haz queda recortado por el umbral, el ancho se **subestima** o no se puede medir.


</details>

---
### ✅ Fin del Laboratorio 3

**Antes de continuar**, guarda tu trabajo: *Archivo → Guardar una copia en Drive* (Colab) o *File → Save* (Jupyter).

➡️ **Siguiente:** **Laboratorio 4 · Escenario multi-emisor** (separar pulsos de varios radares intercalados: *deinterleaving*).

<span style="font-size:11px;color:#7a7a7a">Curso de Guerra Electrónica · Análisis de Señales de Radar · Material de instrucción con datos sintéticos.</span>